In [1]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2, os, shutil, glob, json
import numpy as np
import pandas as pd
import torch.nn as nn
import seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', None)

In [2]:
OPTIONS = json.loads(open('../../Task/info.json', 'r').read())
OPTIONS

{'img_size': [4, 128, 128],
 'step': 3,
 'network': 'unet3d_v2',
 'lr': 0.0001,
 'loss': 'dice_focal',
 'batch_size': 4,
 'scheduler': 'plateau',
 'dropout': 0.1,
 'num_filters': 16}

In [3]:
IMG_SIZE = tuple(OPTIONS.get('img_size'))
IMG_SIZE

(4, 128, 128)

In [4]:
def loadFile(path, size=(128, 128, 128)):
    file = Path(path)
    ext  = file.suffix.lower()

    if ext == '.png':
        return cv2.imread(path)
    
    if ext == '.npy':
        return np.load(path)
    
    if ext == '.dat':
        return np.reshape(np.fromfile(path, dtype=np.single), size)

    return None

def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]

In [5]:
images = [loadFile(img, IMG_SIZE) for img in getFiles('images')]
masks  = [loadFile(img, IMG_SIZE) for img in getFiles('masks')]

IMG_SIZE = images[0].shape[0]
print(len(images), IMG_SIZE)
images[:5]

22270 4


[array([[[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.45398232,
          0.45757064, 0.46159425],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.44910604,
          0.4527773 , 0.45565534],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.45219803,
          0.457582  , 0.46895623],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.49723858,
          0.4920319 , 0.48441008],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.4827063 ,
          0.47786868, 0.47352067],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.47623354,
          0.47256964, 0.47472328]],
 
        [[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.46095577,
          0.4642003 , 0.4650347 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.45365933,
          0.45661938, 0.4621381 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.45378688,
          0.46045375, 0.47266814],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5016018 ,
          0.4961139 , 0.

In [6]:
def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

setFolder('../../Model/Database/Target/images')
setFolder('../../Model/Database/Target/masks')

for i, (img, mask) in enumerate(zip(images, masks)):
    np.save(f'../../Model/Database/Target/images/img_{i:04d}.npy', img)
    np.save(f'../../Model/Database/Target/masks/img_{i:04d}.npy', mask)